# 08. Biological Validation

This notebook upgrades internal biological diagnostics to external validation. The goal is to determine whether predicted perturbation responses recover known functional programs rather than only reproducing within-dataset rankings. The notebook expects one or more pathway gene-set files to be placed under `data/external/`, for example MSigDB Hallmark, Reactome, GO Biological Process, or KEGG GMT files.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 46.0 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [5]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [7]:
!pip -q install gseapy
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from pathlib import Path

try:
    import gseapy as gp
except:
    !pip -q install gseapy
    import gseapy as gp

OUT = RESULTS_DIR / "canonical"

deg_file = OUT / "deg_auprc_per_sample.csv"

assert deg_file.exists(), f"Missing file: {deg_file}"

deg_df = pd.read_csv(deg_file)

print("Loaded:", deg_df.shape)
print("Using file:", deg_file)

Loaded: (60393, 6)
Using file: /content/drive/MyDrive/ChemDFM/results/canonical/deg_auprc_per_sample.csv


## External resource discovery

The notebook scans the external data directory for GMT files. If no GMT files are present, the notebook still runs the bookkeeping cells but the enrichment analysis sections will be skipped. This makes the workflow explicit: biological validity requires external references.


In [8]:
gmt_files = sorted(EXTERNAL_DIR.rglob("*.gmt"))
pd.DataFrame({"gmt_file": [str(p) for p in gmt_files]})


,gmt_file


In [9]:
# User action: choose one or more GMT files once available.
SELECTED_GMT = str(gmt_files[0]) if len(gmt_files) else None
print("SELECTED_GMT =", SELECTED_GMT)


SELECTED_GMT = None


## Ranked-gene enrichment concordance

The recommended analysis is to compute enrichment on true and predicted ranked gene lists for matched perturbation samples or sample groups, and then compare normalized enrichment scores or pathway ranks. This produces a stronger pathway-level validation than internal grouping alone.


In [10]:
if gp is None or SELECTED_GMT is None:
    print("Skipping enrichment because gseapy or GMT resource is unavailable.")
else:
    # Placeholder workflow skeleton: replace `gene_rank_df` with ranked genes per sample or per pseudobulk group.
    # The exact gene list should be generated by reloading 07 outputs together with the source adata.var gene names.
    print("Enrichment resources are available. Implement per-sample or per-group prerank analysis here.")


Skipping enrichment because gseapy or GMT resource is unavailable.


In [11]:
# Optional benchmark metadata hook
benchmark_metadata = list((EXTERNAL_DIR / "benchmark_metadata").glob("*"))
pd.DataFrame({"benchmark_resource": [str(p) for p in benchmark_metadata]})


,benchmark_resource


## Outputs

Recommended outputs from this notebook include:

- `gsea_true_vs_pred_summary.csv`
- `pathway_enrichment_concordance.csv`
- `known_signature_examples.csv`
- `moa_consistency_report.csv`

These files should be saved under `results/biological_validation/` so that the paper notebook can collect them later.


In [12]:
(RESULTS_DIR / "biological_validation").mkdir(parents=True, exist_ok=True)
print("Biological validation notebook scaffold ready.")


Biological validation notebook scaffold ready.
